<!-- notebook-header -->
# Fine-tuning de Modelos Generativos

**Modulo:** 05 - Dominios Aplicados / 05C - Generative AI  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Transfer learning, LoRA, adapters, datasets, avaliacao e riscos de overfitting.


# Fine-tuning de Modelos Generativos: Adaptando Modelos Pre-treinadosObjetivo: Dominar tecnicas de fine-tuning para LLMs, Diffusion Models e Vision Models.Duracao: 120-150 minutos

## 1. Introducao: Por Que Fine-tuning?### Analogia do Carro de CorridaOpcao A (Construir do zero): 2 anos, 10 milhoesOpcao B (Ajustar existente): 2 semanas, 100 milFine-tuning = Opcao B para IA!### O que eh Fine-tuning?Adaptar modelo pre-treinado para tarefa especifica com seus dados.### O que observar em fine-tuning:1. Modelo captura patterns genericos pre-treino2. Seu dominio pode diferir significativamente3. Poucos dados causam risco overfitting4. Ajuste deve ser gentil para preservar features5. Taxa aprendizado deve ser menor que treino zero### O que concluir sobre fine-tuning:1. Fine-tuning economiza tempo exponencial2. Fine-tuning economiza dados necessarios3. Fine-tuning eh standard pratica industria4. Trade-off: menos flexibilidade que treinar zero5. Depende fortemente de qualidade modelo base### Conexao com outros notebooks:1. Transfer Learning fundamentos: 4_4_transfer_learning.ipynb2. Otimizacao e gradientes: 0_8_otimizacao_ml.ipynb3. Probabilidade e estatistica: 0_6 e 1_24. Arquiteturas deep: 4_2_arquiteturas_deep.ipynb5. Validacao cruzada: 3_1_classificacao_completa.ipynb### Por que em ML:1. Dados rotulados sao recurso escasso2. Modelos pre-treinados sao gigantescos3. Fine-tuning = leverage exponencial4. Quase ninguem treina BERT do zero5. 99% de LLM usage eh via fine-tuningPor que em ML: Dados sao recurso mais escasso, reutilizar conhecimento pre-treinado eh imperative.Por que em ML: Comunidade pratica nunca treina modelos baseline do zero.Por que em ML: Dados caros, leverage exponencial.

## 2. Transfer Learning: Conceito Fundamental### Ideia: Reutilizar Features AprendidasFeatures hierarquicas:- Camada 1: Arestas (uteis sempre)- Camada 5: Texturas- Camada 10: Partes (olhos, nariz)- Camada 15: Conceitos (rosto)### O que observar em transfer learning:1. Convergencia eh 5-10x mais rapida2. Loss final melhor com poucos dados3. Com muitos dados, ambas convergem4. Bons pesos iniciais sao cruciais5. Nao precisam aprender features basicas### O que concluir sobre transfer learning:1. Transfer learning funciona por reutilizacao2. Convergencia exponencialmente mais rapida3. Qualidade melhor especialmente com dados escassos4. Fundamental para ML moderno5. Irresponsavel ignorar transfer learning### Conexao com outros notebooks:1. Algoritmo gradient descent: 0_8_otimizacao_ml.ipynb2. Convergencia estocastica: 1_2_estatistica_inferencial.ipynb3. Redes neurais basico: 4_1_fundamentos_redes_neurais.ipynb4. Arquiteturas: 4_2_arquiteturas_deep.ipynb5. Exemplos praticos de transfer: 4_4_transfer_learning.ipynb### Por que em ML:1. Reducao dados necessarios2. Reducao tempo treinamento3. Reducao computacao necessaria4. Reducao riscos de overfitting5. Modelo base captura conhecimento universalPor que em ML: Transfer learning eh foundation de ML moderno pratico.Por que em ML: Economias exponenciais em dados e tempo sao decisivas.Por que em ML: Dados sao recurso escasso, leverage pre-treinado eh economicamente impactful.Por que em ML: Reutilizar eficiente, fundamental.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Simulacao: Feature Extraction vs Full Fine-tuning
np.random.seed(42)
# Cenario com diferentes tamanhos de dataset
dataset_sizes = [50, 100, 200, 500, 1000, 2000]
feature_extraction_acc = [0.65, 0.70, 0.75, 0.80, 0.82, 0.83]
full_finetuning_acc = [0.60, 0.68, 0.78, 0.85, 0.88, 0.90]
plt.figure(figsize=(10, 6))
plt.plot(
dataset_sizes, 
feature_extraction_acc, 'o-', label='Feature Extraction', linewidth=2, markersize=8)
plt.plot(
dataset_sizes, 
full_finetuning_acc, 's-', label='Full Fine-tuning', linewidth=2, markersize=8)
plt.xlabel('Tamanho Dataset')
plt.ylabel('Acurácia')
plt.title('Feature Extraction vs Full Fine-tuning')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.savefig('/tmp/fe_vs_finetuning.png', dpi=100, bbox_inches='tight')
plt.show()
print('Análise: Feature Extraction melhor com poucos dados')
print('Conclusão: Use FE para <500 exemplos, Full para >1000')

In [ ]:
# Feature Extraction vs Full Fine-tuning
np.random.seed(42)
dataset_sizes = [50, 100, 200, 500, 1000, 2000]
feature_extraction_acc = [0.65, 0.70, 0.75, 0.80, 0.82, 0.83]
full_finetuning_acc = [0.60, 0.68, 0.78, 0.85, 0.88, 0.90]

plt.figure(figsize=(10, 6))
plt.plot(dataset_sizes, feature_extraction_acc, "o-", label="Feature Extraction", linewidth=2, markersize=8)
plt.plot(dataset_sizes, full_finetuning_acc, "s-", label="Full Fine-tuning", linewidth=2, markersize=8)
plt.xlabel("Tamanho Dataset")
plt.ylabel("Acuracia")
plt.title("Feature Extraction vs Full Fine-tuning")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale("log")
plt.tight_layout()
plt.savefig("/tmp/fe_vs_finetuning.png", dpi=100, bbox_inches="tight")
plt.show()
print("Analise: Feature Extraction melhor com poucos dados")
print("Conclusao: Use FE para <500 exemplos, Full para >1000")


## 3. Full Fine-tuning vs Feature Extraction### Estrategia A: Feature Extraction- Congela modelo pre-treinado- Treina apenas ultimas camadas- Menos parametros, dados, compute### Estrategia B: Full Fine-tuning- Atualiza todos os pesos- Maxima flexibilidade- Precisa mais dados e compute### O que observar em ambas estrategias:1. n=50 dados: Feature Extraction converge melhor2. n=500 dados: Full fine-tuning comeca vencer3. n=1000+ dados: Full fine-tuning claramente melhor4. Feature Extraction eh mais suave5. Feature Extraction tem menor variancia### O que concluir sobre escolha:1. Menos 500 dados: Use Feature Extraction2. Mais 1000 dados: Use Full Fine-tuning3. 500-1000 dados: Use hibrido (FE depois Full)4. LR: Full Fine-tuning precisa 10-100x menor5. Regularizacao: Full Fine-tuning precisa maior### Conexao com outros notebooks:1. Regularizacao conceitos: 0_8_otimizacao_ml.ipynb2. Early stopping: 3_1_classificacao_completa.ipynb3. Bias-variance tradeoff: 1_2_estatistica_inferencial.ipynb4. Validation splitting: 2_2_eda_completa.ipynb5. Metricas avaliacao: 3_2_regressao_modelos.ipynb### Por que em ML:1. Trade-off bias-variance fundamental2. Balance entre flexibilidade e regularizacao3. Dados determinam viabilidade4. Hardware limita abordagem5. Empiricamente validado repeatedamentePor que em ML: Trade-off bias-variance eh central em machine learning.Por que em ML: Empiricamente validado que balance eh chave.Por que em ML: Bias-variance central, balance importante.

## 4. LoRA: Low-Rank Adaptation### Problema: Memoria InsuficienteLLAMA-7B: 7 bilhoes parametrosFull fine-tuning: 100 GB memoria necessariaGPU consumer: 16 GB disponivelSolucao: LoRA (Low-Rank Adaptation)### Ideia: Rank DecompositionMudanca de pesos tem rank baixo!Delta_W = A @ B- A: d x rank- B: rank x dEconomia: 50-100x parametros!### O que observar em LoRA:1. Captura mudanca com rank baixo2. Convergencia pratica identica3. Qualidade final praticamente igual4. Economia memoria: 50-100x5. Rank 8-64 eh sweet spot pratico### O que concluir sobre LoRA:1. LoRA eh breakthrough tecnologico2. Acesso democratizado agora3. Multiplas LoRAs podem ser combinadas4. Default moderno para LLMs5. Pesquisa continua inovando### Conexao com outros notebooks:1. Algebra linear: 0_3_algebra_linear_matrizes.ipynb2. Matrix decomposition: 0_2_algebra_linear_vetores.ipynb3. SVD e rank: 0_3_algebra_linear_matrizes.ipynb4. Otimizacao subespaço: 0_8_otimizacao_ml.ipynb5. Aplicacao em transformers: 5B_4_transformers_bert.ipynb### Por que em ML:1. Revolução 2023-2024 em LLM2. Permitiu fine-tuning em GPUs consumer3. Transformou economicamente field4. Comprovado funcionar empiricamente5. Agora standard industriaPor que em ML: Memoria eh bottleneck pratico em producao.Por que em ML: Low-rank assumption empiricamente correto em pratica.Por que em ML: Transfer learning fundamenta qualquer fine-tuning bem-sucedido.Conexao com outros notebooks: Fundamento em 4_4_transfer_learning.ipynbPor que em ML: Memoria bottleneck critico, rank funciona.

In [ ]:
# Impacto da Learning Rate
np.random.seed(42)
learning_rates = [0.1, 0.01, 0.001, 0.0001, 0.00001]
training_losses = {
    0.1: [float("nan"), float("nan"), float("nan")] + list(np.linspace(0.5, 0.4, 30)),
    0.01: list(np.linspace(0.8, 0.3, 20)) + list(np.linspace(0.3, 0.25, 13)),
    0.001: list(np.linspace(0.7, 0.15, 20)) + list(np.linspace(0.15, 0.08, 13)),
    0.0001: list(np.linspace(0.65, 0.10, 20)) + list(np.linspace(0.10, 0.05, 13)),
    0.00001: list(np.linspace(0.65, 0.40, 20)) + list(np.linspace(0.40, 0.35, 13))
}

epochs = np.arange(33)
plt.figure(figsize=(10, 6))
colors = ["red", "orange", "green", "blue", "purple"]

for (lr, losses), color in zip(training_losses.items(), colors):
    valid_losses = [l for l in losses if not np.isnan(l)]
    plt.plot(range(len(valid_losses)), valid_losses, label=f"LR={lr}", linewidth=2, color=color)

plt.xlabel("Epoca")
plt.ylabel("Loss de Validacao")
plt.title("Impacto da Learning Rate em Fine-tuning")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/learning_rate_impact.png", dpi=100, bbox_inches="tight")
plt.show()
print("Observacao: LR=0.001 a 0.0001 sao ideais para fine-tuning")
print("Erro comum: Usar LR=0.1 (destroi modelo pre-treinado)")


## 5. QLoRA: Quantizacao + LoRA### Problema Restante: Memoria Modelo BaseLoRA reduz parametros treinaveisMas modelo base ainda precisa memoriaSolucao: Quantizar modelo base### Como QLoRA Funciona1. Carregar modelo em int4 (4 bits)2. Durante treino, desquantizar para float163. Atualizar apenas LoRA weights4. Gradientes fluxam atraves modelo5. Salvar checkpoint: apenas LoRAResultado: 16GB memoria -> 6GB memoria!### O que observar em QLoRA:1. Trade-off muito favoravel2. int4 eh balance point otimo3. Permite impossivel sem QLoRA4. Custo ~50% vs LoRA puro5. Degradacao qualidade minuscula (<2%)### O que concluir sobre QLoRA:1. Democratizacao de fine-tuning2. Modelo 70B cabe em 16GB3. Qualidade praticamente intacta4. Recomendado para hardware limitado5. Combinacao com Flash Attention=otimo### Conexao com outros notebooks:1. Quantizacao teoria: 4_5_aceleracao_hardware.ipynb2. Precisao numerica: 0_2_algebra_linear_vetores.ipynb3. Memory management: 4_5_aceleracao_hardware.ipynb4. Hardware acceleration: 4_5_aceleracao_hardware.ipynb5. Inference optimization: 6_1_deploy_modelos.ipynb### Por que em ML:1. Hardware crescimento lag vs modelo tamanho2. Quantizacao eh ponte necessaria3. Economicamente transformador4. Democratiza big model acesso5. Pratica agora ubiquaPor que em ML: Hardware crescimento nao acompanha tamanho modelo.Por que em ML: Quantizacao eh bridge tecnica necessaria.Por que em ML: Hardware lag existe, quantizacao necessaria.

In [ ]:
# LoRA: Low-Rank Adaptation
np.random.seed(42)
d = 1000
W_base = np.random.randn(d, d) * 0.01
rank = 16
U = np.random.randn(d, rank) * 0.01
V = np.random.randn(rank, d) * 0.01
Delta_W_true = U @ V
params_full = d * d
params_lora = 2 * d * rank
print(f"Full fine-tuning: {params_full:,} parametros")
print(f"LoRA (rank={rank}): {params_lora:,} parametros")
print(f"Economia: {params_full / params_lora:.1f}x")


## 6. Fine-tuning de Large Language Models### Modelos Base: Genericos- Treinados em internet balanceada- Conhecimento amplo mas superficial- Sem especialização tarefa- Requerem fine-tuning para producao### O que observar em LLM fine-tuning:1. Tamanho base determina capacidade2. Qualidade dados > Quantidade dados3. Learning rate extremamente critico4. Overfitting facil com 1000 exemplos5. 3-5 epochs tipicamente suficiente### O que concluir sobre LLM fine-tuning:1. Fine-tuned models >> base models2. Accessivel via LoRA/QLoRA3. Horas treino, minutos inference4. Dados bons importam mais5. Baseline comparacao essencial### Conexao com outros notebooks:1. Transformers: 5B_4_transformers_bert.ipynb2. LLMs especifico: 5B_5_large_language_models.ipynb3. Prompt engineering: 5B_5_large_language_models.ipynb4. Embeddings: 5B_2_word_embeddings.ipynb5. RAG tecnicas: 5B_6_rag_aplicacoes_llm.ipynb### Por que em ML:1. LLMs sao powerful mas generic2. Fine-tuning especializa para dominio3. Ponte entre general e specific4. Acessivel agora5. Transforma LLM em asset utilPor que em ML: LLMs genericos requerem especialização pratica.Por que em ML: Fine-tuning transforma modelo generic em asset util.Por que em ML: Balanceamento bias-variance eh central.Conexao com outros notebooks: Trade-offs explorados em 1_2_estatistica_inferencial.ipynbPor que em ML: LLMs genericos, especializacao essencial.

## 7. Fine-tuning de Diffusion Models### Diferenca de Modelos GenerativosLLM: Comportamento textual, cross-entropy lossDiffusion: Qualidade visual, MSE de ruidoTecnicas: DreamBooth, Textual Inversion### DreamBooth: Fine-tune Objeto Especifico- Input: 5-30 imagens do objeto- Output: Modelo gera infinitas variacoes- Qualidade: Professional grade### O que observar em diffusion fine-tuning:1. DreamBooth muito eficaz poucos dados2. Textual Inversion eh mais rapido3. Convergencia em 10-100 steps4. 5-30 imagens eh sweet spot5. Variacao pose/iluminacao importante### O que concluir sobre diffusion fine-tuning:1. Revolucionou customizacao imagem2. Accessivel (nao precisa 10k imagens)3. Qualidade profissional alcancada4. Minutos vs horas vs dias5. Democratizou geracao customizada### Conexao com outros notebooks:1. Diffusion models: 5C_3_diffusion_models.ipynb2. GANs alternativa: 5C_1_gans.ipynb3. VAEs alternativa: 5C_2_vaes.ipynb4. Noise scheduling: 5C_3_diffusion_models.ipynb5. Sampling tecnicas: 5C_3_diffusion_models.ipynb### Por que em ML:1. Geracao customizada demanda massiva2. DreamBooth transformou campo3. Ganhou notoriedade mainstream4. Adobe/Runway integrados5. Futuro: real-time fine-tuningPor que em ML: Customizacao eh demanda principal usuarios.Por que em ML: Accessibility democratiza technology.Por que em ML: Customizacao demanda, democratizacao importante.

## 8. Dataset Curation: Fundamento Critico### Garbage In, Garbage Out: Lei FundamentalMelhor modelo + pior dataset = modelo ruimModelo ok + excelente dataset = excelente modeloDataset = 60-70% do sucesso!### O que observar em dataset curation:1. Duplicatas causam overfitting imediato2. Vieses dados se herdam no modelo3. Ruido extremo prejudica muito4. Distribuicao treino != producao5. Anotacoes inconsistentes destroem signal### O que concluir sobre curation:1. Qualidade primordial sempre2. Tempo em curation economiza iteracoes3. 80% tempo em dados, 20% em modelo4. Spot-check sempre amostras5. Versionamento essencial rastreabilidade### Conexao com outros notebooks:1. EDA exploracao: 2_2_eda_completa.ipynb2. Limpeza dados: 2_1_python_data_science.ipynb3. APIs coleta: 2_3_sql_e_apis.ipynb4. Balanceamento classes: 3_1_classificacao_completa.ipynb5. Data quality: 2_4_acesso_banco_dados.ipynb### Por que em ML:1. Dados sao recurso mais escasso2. Coleta eh bottleneck pratica3. Qualidade determina ceiling4. Vieses replicam em modelo5. Responsabilidade etica criticaPor que em ML: Data quality eh determinante absoluto.Por que em ML: Garbage data gera garbage model sempre.Por que em ML: Memoria eh hardware bottleneck critico.Conexao com outros notebooks: Algebra linear decomposition em 0_3Por que em ML: Dados determinam ceiling, qualidade primaria.

## 9. Avaliacao: Metricas e Deteccao### O Que Medir? Multiplas DimensoesMetricas genericas: Train loss, Val loss, Early stoppingMetricas especificas: Accuracy, F1, BLEU, mAP, Perplexidade### O que observar em avaliacao:1. Curva treino muito reveladora2. Validacao metricas devem ser realistas3. Problemas numericos aparecem cedo4. Metricas dominio mais importante5. Avaliacao qualitativa eh invaluavel### O que concluir sobre monitoring:1. Quantitativo necessario mas insuficiente2. Overfitting eh risco numero 13. Combinacao quantitativo+qualitativo melhor4. Early stopping salva tempo5. Baseline comparacao eh essencial### Conexao com outros notebooks:1. Metricas classificacao: 3_1_classificacao_completa.ipynb2. Validacao cruzada: 1_2_estatistica_inferencial.ipynb3. Matriz confusao: 3_1_classificacao_completa.ipynb4. Statistical testing: 1_2_estatistica_inferencial.ipynb5. ROC curves: 3_1_classificacao_completa.ipynb### Por que em ML:1. Metricas sao feedback loop2. Sem metricas, cegueira completa3. Metricas guiam investigacao4. Erros detectaveis via metricas5. Reproducibilidade via loggingPor que em ML: Metricas sao feedback loop essencial.Por que em ML: Sem metricas, desenvolvimento eh cecidade.

## 10. Exercicios Praticos### Exercicio 1: Fine-tuning SimulatorImplementar simulator mostrando impacto hiperparametros

Por que em ML: Hardware crescimento lag modelo tamanho.
Conexao com outros notebooks: Quantizacao em 4_5_aceleracao_hardware.ipynb

In [ ]:
# EXERCICIO 1: Fine-tuning Simulator
# TAREFA DO ALUNO: Completar funcao
def fine_tune_simulator(learning_rate=0.001, regularization=0.0001, n_epochs=30, n_data=100):    pass
print('Exercicio 1: Implemente acima')

### Exercicio 2: Comparacao EstrategiasComparar FE vs LoRA em diferentes dataset sizes

In [ ]:
# EXERCICIO 2
# TAREFA DO ALUNO: Completar
def compare_strategies(n_data_values=[50, 100, 500, 1000]):    pass
print('Exercicio 2: Implemente')

### Exercicio 3: Dataset ChecklistCriar checklist qualidade para dataset

In [ ]:
# EXERCICIO 3# TAREFA DO ALUNO: Preencher checklistdataset_checklist = {}print('Exercicio 3: Complete')

## 11. Erros Comuns em Fine-tuning### Erro 1: Nao Usar Early StoppingProblema: Treinar N epochs mesmo com overfittingSolucao: Monitorar validacao, parar quando piora### Erro 2: Learning Rate ErradoProblema: LR 0.1 (treino zero) destroi fine-tuningSolucao: 10-100x menor (0.001 a 0.0001)### Erro 3: Sem Split ValidacaoProblema: Nao isolou validacao/testeSolucao: 80/10/10 split sempre### Erro 4: Dataset Muito PequenoProblema: <50 exemplos com Full Fine-tuningSolucao: Feature Extraction ou LoRA### Erro 5: Ignorar Imbalance ClassesProblema: 90% classe A, 10% classe BSolucao: Class weighting ou oversampling### O que observar sobre erros:1. Learning rate causa maioria problemas2. Data leakage invalida resultados3. Imbalance causa surpresas producao4. Pequeno dataset garante overfitting5. Sem early stopping desperdica tempo### O que concluir:1. LR mais importante que outro parametro2. Data isolation nao negociavel3. Validacao seu maior aliado4. Early stopping economiza tempo5. Disciplina eh preventivo

### Hierarquia de Conceitos**Nível 0: Fundamentação**- Probabilidade e Estatística (0_6, 1_2)- Otimização ML (0_8)- Redes Neurais Básicas (4_1)**Nível 1: Arquiteturas Base**- Deep Learning Arquiteturas (4_2)- Transformers e BERT (5B_4)- LLMs (5B_5)**Nível 2: Métodos Generativos**- GANs (5C_1)- VAEs (5C_2)- Diffusion Models (5C_3)**Nível 3: Especialização (Fine-tuning)**- Transfer Learning (4_4)- Fine-tuning LLMs com LoRA/QLoRA- Fine-tuning Diffusion (DreamBooth)**Nível 4: Integração Avançada**- Multimodal AI (5C_5)- Aplicações RAG (5B_6)**Nível 5: Produção**- Deployment Modelos (6_1)- Monitoramento (6_3)### Próximos Passos**Imediato:**1. Implementar fine-tuning com LoRA em PyTorch2. Treinar em dataset próprio3. Comparar resultados com baseline**Curto Prazo:**4. Implementar QLoRA para modelos maiores5. Explorar DreamBooth para diffusion6. Validar em dados de produção**Médio Prazo:**7. Otimizar para inferência8. Integrar em aplicação real9. Monitorar performance**Longo Prazo:**10. Contribuir para open source11. Pesquisar novos métodos de fine-tuning12. Escalar para múltiplos domínios

## 12. Resumo Hierarquico e Proximos Passos### Nivel 0: Core ConceptFine-tuning = Adaptar pre-treinado para dominioVantagens: 10-100x economia dados, tempo, compute### Nivel 1: Estrategias- Feature Extraction: Poucos dados- Full Fine-tuning: Muitos dados- LoRA: Recomendado (balance optimo)- QLoRA: Hardware limitado### Nivel 2: Implementacao Prática1. Escolher modelo base2. Preparar dataset (limpar, balancear)3. Escolher estrategia (LoRA recomendado)4. Configurar hiperparametros5. Treinar com monitoramento6. Avaliar quantitativo+qualitativo7. Iterar com feedback### Nivel 3: Aplicacoes Especificas- LLMs: LoRA, QLoRA, Full Fine-tuning- Diffusion: DreamBooth, Textual Inversion- Vision: Feature Extraction, Full Fine-tuning### Checklist Implementacao Completa- [ ] Dataset preparado e limpo- [ ] Split treino/val/teste isolados- [ ] Modelo base escolhido- [ ] Estrategia decidida- [ ] Hiperparametros configurados- [ ] Learning rate validado- [ ] Early stopping implementado- [ ] Metricas dominio definidas- [ ] Baseline estabelecido- [ ] Training curves monitoradas- [ ] Avaliacao qualitativa feita- [ ] Comparacao vs baseline ok- [ ] Modelo salvo e versionado- [ ] Documentacao escrita### Conexoes Completas com Outros Notebooks- Probabilidade: 0_6 Probabilidade- Otimizacao: 0_8 Otimizacao ML- Estatistica: 1_2 Estatistica Inferencial- Redes Neurais: 4_1 Fundamentos, 4_2 Arquiteturas- Transfer Learning: 4_4 Transfer Learning- LLMs: 5B_4 Transformers, 5B_5 LLMs- Diffusion: 5C_3 Diffusion Models- Deploy: 6_1 Deploy Modelos### Por Que Em Machine Learning (10 razoes fundamentais)1. Realidade industrial: 100% de proyectos usam fine-tuning2. Eficiencia economica: Melhor ROI que treino zero3. Democratizacao: PMEs podem treinar LLMs agora4. Ciclos rapidos: Inovacao mais veloz5. Dominio critico: Essencial para sucesso6. Dados escassos: Realidade pratica always7. Hardware lag: GPU nao cresce rapido como modelos8. Economia massiva: 1000x mais barato que zero9. Pragmatismo: Funciona empiricamente proven10. Futuro: Sera ainda mais central### Conclusao FinalFine-tuning eh technologia MADURA e ESSENCIAL em 2024+.Combinacao unica:- Modelos pre-treinados poderosos (GPT, LLAMA, Stable Diffusion)- Tecnicas eficientes (LoRA, QLoRA, DreamBooth)- Ferramentas acessiveis (HuggingFace, LLaMA-Factory)Resultado: Qualquer pessoa adapta modelos state-of-the-art para seu caso de uso.Mensagem final: Entenda principios fundamentais. Implemente com disciplina. Itere com dados de qualidade. Sucesso seguira naturalmente.

## Soluções dos Exercícios

In [ ]:
# SOLUCAO - Exercicio 1: Fine-tuning Simulator
import numpy as np
np.random.seed(42)

def fine_tune_simulator(n_data, n_epochs=50, lr=0.01, freeze_ratio=0.0):
    d = 100
    W_pretrained = np.random.randn(d, d) * 0.1
    W = W_pretrained.copy()
    X = np.random.randn(n_data, d)
    y = np.sin(X[:, 0]) + 0.1 * np.random.randn(n_data)
    n_freeze = int(d * freeze_ratio)
    losses = []
    for epoch in range(n_epochs):
        pred = X @ W[:, 0]
        loss = np.mean((pred - y) ** 2)
        losses.append(loss)
        grad = 2 * X.T @ (pred - y) / n_data
        W[:, 0] -= lr * grad
        W[:n_freeze, 0] = W_pretrained[:n_freeze, 0]
    return losses

for n in [50, 200, 1000]:
    losses = fine_tune_simulator(n)
    print(f'n_data={n}: loss_final={losses[-1]:.4f}')

In [ ]:
# SOLUCAO - Exercicio 2: Comparar Estrategias
import numpy as np
np.random.seed(42)

def compare_strategies(n_data, d=100):
    X = np.random.randn(n_data, d)
    y = np.sin(X[:, 0]) + 0.1 * np.random.randn(n_data)
    results = {}
    # Feature Extraction
    W_fe = np.zeros(d)
    for epoch in range(100):
        pred = X @ W_fe
        grad = 2 * X.T @ (pred - y) / n_data
        W_fe -= 0.01 * grad
    results['feature_extraction'] = np.mean((X @ W_fe - y) ** 2)
    # Full Fine-tuning
    W_full = np.random.randn(d) * 0.01
    for epoch in range(100):
        pred = X @ W_full
        grad = 2 * X.T @ (pred - y) / n_data
        W_full -= 0.01 * grad
    results['full_finetuning'] = np.mean((X @ W_full - y) ** 2)
    return results

for n in [50, 500]:
    r = compare_strategies(n)
    print(f'n={n}: FE={r["feature_extraction"]:.4f} Full={r["full_finetuning"]:.4f}')

In [ ]:
# SOLUCAO - Exercicio 3: Dataset Checklist
dataset_checklist = {
    'tamanho_adequado': True,
    'qualidade_labels': True,
    'balanceamento_classes': False,
    'sem_data_leakage': True,
    'formato_consistente': True,
    'preprocessing_necessario': True,
    'split_train_val_test': True
}

print('Dataset Checklist para Fine-tuning:')
print('=' * 50)
for item, status in dataset_checklist.items():
    marca = 'OK' if status else 'ATENCAO'
    print(f'  {item}: [{marca}]')
n_ok = sum(v for v in dataset_checklist.values())
n_total = len(dataset_checklist)
print(f'Score: {n_ok}/{n_total}')

In [ ]:
# QLoRA: Quantizacao + LoRA
np.random.seed(42)
model_size_mb_float32 = 7000 * 4 / 1024 / 1024
model_size_mb_int4 = 7000 * 0.5 / 1024 / 1024
rank = 8
lora_size_gb = (7000 * rank * 2 * 4) / (1024 * 1024 * 1024)

print(f"Modelo base (float32): {model_size_mb_float32:.0f} MB")
print(f"Modelo base (int4): {model_size_mb_int4:.0f} MB")
print(f"LoRA overhead: {lora_size_gb:.2f} GB")
print(f"Total QLoRA: {model_size_mb_int4/1024 + lora_size_gb:.2f} GB")


In [ ]:
# Visualizacao QLoRA
categories = ["Float32\n(Full)", "Int4\n(Quantizado)", "QLoRA\n(Quantizado + LoRA)"]
memory_gb = [28, 1.75, 3.5]

plt.figure(figsize=(8, 5))
colors_bar = ["red", "orange", "green"]
bars = plt.bar(categories, memory_gb, color=colors_bar, alpha=0.7, edgecolor="black", linewidth=2)
plt.ylabel("Memoria (GB)")
plt.title("QLoRA: Reducao de Memoria para Fine-tuning de LLMs")
plt.axhline(y=16, color="black", linestyle="--", label="GPU Consumer (16GB)", linewidth=2)
plt.legend()
for bar, val in zip(bars, memory_gb):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{val:.1f}GB", ha="center", va="bottom", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/qlora_memory.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
# Demonstracao sazonalidade ignorada
t = np.arange(120)
y_saz = 100 + 10*np.sin(2*np.pi*t/12) + np.random.normal(0, 1, 120)
plt.figure(figsize=(12, 4))
plt.plot(t, y_saz, "b-", linewidth=1.5)
plt.title("Serie com Sazonalidade Importante")
plt.xlabel("Tempo (meses)")
plt.ylabel("Valor")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/ignorar_sazonalidade.png", dpi=100, bbox_inches="tight")
plt.show()
print("Erro: Ignorar sazonalidade causa bad generalization")
